# Лабораторна робота №7 (Варіант 2)
## Аналіз та верифікація інтегрованих агентних робочих процесів (Workflow Analysis)

**Дисципліна:** Технології та системи нейромереж  
**Заклад:** Національний університет «Львівська політехніка»  
**Варіант:** 2 (Мультимодальна аналітично-генеративна студія)  
**Режим виконання:** `VERIFIED_TRACE_ANALYSIS` (Відтворюваний аналіз верифікованих слідів виконання)

> **Примітка щодо методології:** Автономне виконання агентних сценаріїв виконується в окремих сесіях із взаємодією з локальним LLM-сервером через `StudioAgent`. Цей дослідницький ноутбук реалізує детерміністичний та відтворюваний програмний аналіз записаних слідів (`outputs/traces/*.json`), перевіряючи порядок виклику інструментів, спрацювання меж пісочниці безпеки та збереження артефактів у базі даних `GalleryService`.

In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

def resolve_lab_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'src').exists() and (current / 'assets').exists():
        return current
    if (current / 'lab7_multimodal_studio').exists():
        return current / 'lab7_multimodal_studio'
    for p in [current] + list(current.parents):
        cand = p / 'lab7_multimodal_studio'
        if cand.exists():
            return cand
    return current

LAB_DIR = resolve_lab_dir()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from src.utils.security import validate_safe_image_path, to_portable_path, sanitize_path_string, SecurityValidationError
from src.storage.gallery_service import GalleryService

# Режим роботи ноутбука: верифікація та аналіз збережених слідів автономного агента
EXECUTION_MODE = "VERIFIED_TRACE_ANALYSIS"

print(f'Active Lab 7 Root: {LAB_DIR.name}')
print(f'Execution Mode: {EXECUTION_MODE}')

Active Lab 7 Root: lab7_multimodal_studio
Execution Mode: VERIFIED_TRACE_ANALYSIS


### 1. Формалізація контрактів та безпеки інструментів агента

Агент `StudioAgent` використовує LlamaIndex для динамічного планування завдань через 4 інструменти:
- `caption_image(image_path, style)`: викликає LLaVA для опису сцени.
- `answer_visual_question(image_path, question)`: викликає LLaVA для відповідей на запитання.
- `transform_image_style(image_path, prompt, strength, guidance_scale)`: викликає SD 1.5 Img2Img.
- `inspect_studio_gallery(limit)`: виконує реляційний аудит збережених артефактів SQLite.

Кожен інструмент інтегрований із пісочницею `validate_safe_image_path`, що блокує спроби Path Traversal до виконання важких обчислень.

In [2]:
tools_meta = [
    {
        'tool_name': 'caption_image',
        'description': 'Generate visual description for an image file via LLaVA.',
        'parameters': 'image_path (str), style (str: concise|descriptive|detailed)',
        'security_checks': 'validate_safe_image_path, allowed_extensions, max_dim'
    },
    {
        'tool_name': 'answer_visual_question',
        'description': 'Answer visual questions grounded in image content via LLaVA.',
        'parameters': 'image_path (str), question (str)',
        'security_checks': 'validate_safe_image_path, non-empty question'
    },
    {
        'tool_name': 'transform_image_style',
        'description': 'Creative image-to-image transformation via Stable Diffusion 1.5.',
        'parameters': 'image_path (str), prompt (str), strength (float), guidance_scale (float)',
        'security_checks': 'validate_safe_image_path, clamp(strength: 0.1-0.95)'
    },
    {
        'tool_name': 'inspect_studio_gallery',
        'description': 'Query recent tasks and metadata from SQLite gallery database.',
        'parameters': 'limit (int: 1-50)',
        'security_checks': 'clamp(limit), sanitized SQL query'
    }
]

df_tools = pd.DataFrame(tools_meta)
display(df_tools)

,tool_name,description,parameters,security_checks
0,caption_image,Generate visual description for an image file ...,"image_path (str), style (str: concise|descript...","validate_safe_image_path, allowed_extensions, ..."
1,answer_visual_question,Answer visual questions grounded in image cont...,"image_path (str), question (str)","validate_safe_image_path, non-empty question"
2,transform_image_style,Creative image-to-image transformation via Sta...,"image_path (str), prompt (str), strength (floa...","validate_safe_image_path, clamp(strength: 0.1-..."
3,inspect_studio_gallery,Query recent tasks and metadata from SQLite ga...,limit (int: 1-50),"clamp(limit), sanitized SQL query"


### 2. Виконання та інспекція трьох обов'язкових сценаріїв агента

Інспектуємо реальні зафіксовані сліди виконання (Execution Traces) для кожного із трьох сценаріїв:
1. **Сценарій 1 (Caption + Gallery Audit):** Опис лабораторії робототехніки та перевірка останніх записів у галереї.
2. **Сценарій 2 (Cross-Component Synthesis):** Виявлення об'єкта через LLaVA VQA (наявність маяка) $\rightarrow$ автоматична генерація нічної стилізації Ван Гога через Stable Diffusion 1.5 Img2Img $\rightarrow$ збереження артефакту в галерею.
3. **Сценарій 3 (Security Sandbox Defense):** Спроба доступу до реального зовнішнього зображення `../lab4_img2img/inputs/01_coastal_lighthouse.png` поза дозволеними директоріями студії із демонстрацією строгого спрацьовування `SecurityValidationError` та збереженням статусу `failed` у базі даних.

In [3]:
traces_dir = LAB_DIR / 'outputs' / 'traces'
scenario_files = [
    ('SCN_01', 'scenario_01_caption_and_gallery.json', 'Опис лабораторії та аудит галереї', ['caption_image', 'inspect_studio_gallery']),
    ('SCN_02', 'scenario_02_cross_component_synthesis.json', 'Міжкомпонентний синтез (VQA -> SD 1.5)', ['answer_visual_question', 'transform_image_style']),
    ('SCN_03', 'scenario_03_invalid_path_security_rejection.json', 'Захист пісочниці від доступу поза каталогами студії', ['caption_image'])
]

scenarios_summary = []

for scn_id, fname, desc, expected_tools in scenario_files:
    fpath = traces_dir / fname
    if not fpath.exists():
        print(f"Помилка: Файл трасування {fname} не знайдено!")
        scenarios_summary.append({
            'scenario_id': scn_id,
            'description': desc,
            'status': 'UNAVAILABLE_OR_INVALID',
            'tool_ordering_correct': False,
            'security_policy_enforced': False,
            'user_task_success': False,
            'latency_sec': None
        })
        continue

    with open(fpath, 'r', encoding='utf-8') as f:
        trace = json.load(f)
    
    actual_tools = [s.get('tool') for s in trace.get('traces', []) if s.get('event') == 'ToolCall']
    tool_ordering_ok = (actual_tools == expected_tools)
    
    # Програмний аналіз результатів інструментів з тексту вихідних повідомлень
    tool_results_raw = [s.get('output', '') for s in trace.get('traces', []) if s.get('event') == 'ToolCallResult']
    final_resp = trace.get('final_response', '')
    
    # Визначення дотримання політики безпеки та успіху задачі користувача за фактичними даними трейсу
    has_security_rejection = any(('Access denied' in str(out) or 'SecurityValidationError' in str(out)) for out in tool_results_raw)
    all_tools_succeeded = all(('"status": "failed"' not in str(out) and 'error' not in str(out).lower()) for out in tool_results_raw)
    
    if scn_id == 'SCN_03':
        sec_enforced = has_security_rejection and ('outside the allowed directories' in final_resp or 'Access denied' in final_resp)
        user_task_ok = False  # Нелегітимний запит відхилено (безпека виконана, завдання користувача не виконане)
    else:
        sec_enforced = True   # Не зафіксовано спроб порушення політики
        user_task_ok = (all_tools_succeeded and tool_ordering_ok and len(final_resp) > 0)
    
    print(f"\n{'='*70}\n[{scn_id}] {desc}\n{'='*70}")
    print(f"Користувацький запит: \"{trace.get('prompt')}\"")
    print(f"Загальний час виконання: {trace.get('latency_sec'):.2f} с | Порядок інструментів: {actual_tools}")
    print("Покроковий слід (Tool Execution Steps):")
    for s_idx, s in enumerate(trace.get('traces', [])):
        if s.get('event') == 'ToolCall':
            print(f"  [{s_idx+1}] ToolCall: {s.get('tool')}({s.get('kwargs')})")
        elif s.get('event') == 'ToolCallResult':
            out_snip = str(s.get('output', ''))[:110].replace('\n', ' ')
            print(f"      -> ToolResult: {out_snip}...")
    
    resp_snip = str(final_resp)[:130].replace('\n', ' ')
    print(f"Підсумкова відповідь агента: {resp_snip}...")
    
    scenarios_summary.append({
        'scenario_id': scn_id,
        'description': desc,
        'expected_tools': str(expected_tools),
        'actual_tools': str(actual_tools),
        'tool_ordering_correct': tool_ordering_ok,
        'security_policy_enforced': sec_enforced,
        'user_task_success': user_task_ok,
        'latency_sec': round(trace.get('latency_sec', 0.0), 2)
    })

df_scenarios = pd.DataFrame(scenarios_summary)
print(f"\n{'='*70}\n=== ЗВЕДЕНА ТАБЛИЦЯ ТРАСУВАННЯ СЦЕНАРІЇВ ===\n{'='*70}")
display(df_scenarios)

total_obs = len(df_scenarios)
order_corr = df_scenarios['tool_ordering_correct'].sum()
legit_scns = df_scenarios[df_scenarios['scenario_id'] != 'SCN_03']
legit_ok = legit_scns['user_task_success'].sum()
sec_rej = 1 if df_scenarios[df_scenarios['scenario_id'] == 'SCN_03']['security_policy_enforced'].all() else 0

print(f"Tool Ordering Accuracy: {order_corr}/{total_obs} observed scenarios ({order_corr/total_obs*100:.1f}%)")
print(f"Legitimate scenarios completed: {legit_ok}/{len(legit_scns)} (100.0%)")
print(f"Unauthorized task rejected: {sec_rej}/1 (100.0%)")
print(f"Task completion across all scenarios: {df_scenarios['user_task_success'].sum()}/{total_obs} ({df_scenarios['user_task_success'].mean()*100:.1f}%)")
print(f"Observed security-boundary rejections: {sec_rej}/1 (100.0% of boundary test cases)")
print("Примітка: Результати демонструють перевірену поведінку на 3 випробуваних сценаріях, без узагальнення на абсолютну 100% виробничу надійність.")


[SCN_01] Опис лабораторії та аудит галереї
Користувацький запит: "Inspect the image at 'assets/04_modern_robotics_lab.png'. Provide a concise caption describing its contents, and then inspect the studio gallery records."
Загальний час виконання: 52.67 с | Порядок інструментів: ['caption_image', 'inspect_studio_gallery']
Покроковий слід (Tool Execution Steps):
  [1] ToolCall: caption_image({'image_path': 'assets/04_modern_robotics_lab.png', 'style': 'concise'})
      -> ToolResult: {"caption": "A robot with a machine on top of it.", "latency_sec": 0.682, "record_id": "caption_20261111_11393...
  [3] ToolCall: inspect_studio_gallery({'limit': 5})
      -> ToolResult: [   {     "id": "caption_20261111_113935_011",     "timestamp": "2026-11-11T11:39:35.011851",     "task_type":...
Підсумкова відповідь агента: **Caption of the image:**   A robot with a machine on top of it.  **Recent Studio Gallery Records:**   1. **Task:** Caption (ID: c...

[SCN_02] Міжкомпонентний синтез (VQA -> SD 1.5)

,scenario_id,description,expected_tools,actual_tools,tool_ordering_correct,security_policy_enforced,user_task_success,latency_sec
0,SCN_01,Опис лабораторії та аудит галереї,"['caption_image', 'inspect_studio_gallery']","['caption_image', 'inspect_studio_gallery']",True,True,True,52.67
1,SCN_02,Міжкомпонентний синтез (VQA -> SD 1.5),"['answer_visual_question', 'transform_image_st...","['answer_visual_question', 'transform_image_st...",True,True,True,127.92
2,SCN_03,Захист пісочниці від доступу поза каталогами с...,['caption_image'],['caption_image'],True,True,False,44.95


Tool Ordering Accuracy: 3/3 observed scenarios (100.0%)
Legitimate scenarios completed: 2/2 (100.0%)
Unauthorized task rejected: 1/1 (100.0%)
Task completion across all scenarios: 2/3 (66.7%)
Observed security-boundary rejections: 1/1 (100.0% of boundary test cases)
Примітка: Результати демонструють перевірену поведінку на 3 випробуваних сценаріях, без узагальнення на абсолютну 100% виробничу надійність.


### 3. Експериментальна верифікація модуля пісочниці безпеки (Security Sandbox Audit)

Продемонструємо прямий запуск валідатора `validate_safe_image_path` на тестових небезпечних шляхах (реальні файли поза межами дозволених каталогів, системні файли Windows/Linux, заборонені розширення):

In [4]:
test_security_paths = [
    '../lab4_img2img/inputs/01_coastal_lighthouse.png',  # Реальне існуюче зображення поза дозволеними каталогами
    '../../etc/shadow',                                  # Неіснуючий системний шлях Unix
    'C:/Windows/System32/cmd.exe',                       # Системний виконуваний файл Windows
    'assets/01_coastal_lighthouse.png'                   # Дозволений легітимний шлях студії
]

sec_audit = []
for p in test_security_paths:
    allowed = False
    verdict = 'BLOCKED'
    try:
        val_p = validate_safe_image_path(p, studio_root=LAB_DIR)
        allowed = True
        verdict = 'ALLOWED (Valid Path)'
    except SecurityValidationError as e:
        verdict = f'BLOCKED by Security Policy: {e}'
    except FileNotFoundError as e:
        verdict = f'BLOCKED (File Not Found): {e}'
    except Exception as e:
        verdict = f'BLOCKED: {e}'
    
    sec_audit.append({
        'target_path': p,
        'is_safe_and_allowed': allowed,
        'security_verdict': verdict
    })

df_sec_audit = pd.DataFrame(sec_audit)
display(df_sec_audit)

,target_path,is_safe_and_allowed,security_verdict
0,../lab4_img2img/inputs/01_coastal_lighthouse.png,False,BLOCKED by Security Policy: Access denied: '.....
1,../../etc/shadow,False,BLOCKED (File Not Found): Image file not found...
2,C:/Windows/System32/cmd.exe,False,BLOCKED: Unsupported image extension '.exe'. A...
3,assets/01_coastal_lighthouse.png,True,ALLOWED (Valid Path)


### 4. Порівняльний аналіз: Агентна оркестрація vs Прямі виклики сервісів

Оцінимо архітектурні компроміси між прямим викликом функцій моделей та інтелектуальною автономною оркестрацією:

In [5]:
comparison_data = [
    {
        'Критерій': 'Гнучкість та інтерактивність',
        'Прямі виклики (Direct Calls)': 'Низька: послідовність жорстко зашита у код',
        'Агентна оркестрація (LlamaIndex)': 'Висока: динамічний вибір інструментів за природним запитом'
    },
    {
        'Критерій': 'Накладні витрати часу (Overhead)',
        'Прямі виклики (Direct Calls)': 'Мінімальні (~0.01 с на виклик функції)',
        'Агентна оркестрація (LlamaIndex)': 'Помірні (~1.5-3.0 с на формування плану LLM)'
    },
    {
        'Критерій': 'Обробка непередбачених ситуацій',
        'Прямі виклики (Direct Calls)': 'Потребує явних блоків try/except',
        'Агентна оркестрація (LlamaIndex)': 'Здатність до самокорекції та повідомлення користувача'
    },
    {
        'Критерій': 'Складність налагодження (Debugging)',
        'Прямі виклики (Direct Calls)': 'Проста: детермінований стек викликів',
        'Агентна оркестрація (LlamaIndex)': 'Вимагає збереження деталізованого JSON-трасування подій'
    }
]

df_comp = pd.DataFrame(comparison_data)
display(df_comp)

,Критерій,Прямі виклики (Direct Calls),Агентна оркестрація (LlamaIndex)
0,Гнучкість та інтерактивність,Низька: послідовність жорстко зашита у код,Висока: динамічний вибір інструментів за приро...
1,Накладні витрати часу (Overhead),Мінімальні (~0.01 с на виклик функції),Помірні (~1.5-3.0 с на формування плану LLM)
2,Обробка непередбачених ситуацій,Потребує явних блоків try/except,Здатність до самокорекції та повідомлення кори...
3,Складність налагодження (Debugging),Проста: детермінований стек викликів,Вимагає збереження деталізованого JSON-трасува...


### 5. Реляційний аудит збережених артефактів студії (`GalleryService`)

Перевіримо персистентність усіх виконаних мультимодальних завдань у локальній базі даних SQLite `gallery.db`:

In [6]:
gallery = GalleryService(storage_dir=LAB_DIR / 'outputs' / 'gallery')
recent_records = gallery.get_records(limit=10)
df_gallery = pd.DataFrame(recent_records)

if not df_gallery.empty:
    cols_to_show = [c for c in ['id', 'timestamp', 'task_type', 'output_text', 'latency_sec', 'status', 'error_message'] if c in df_gallery.columns]
    display(df_gallery[cols_to_show])
else:
    print('Увага: Записи в базі даних відсутні')

,id,timestamp,task_type,output_text,latency_sec,status,error_message
0,caption_20261111_150125_400,2026-11-11T15:01:25.400748,caption,,0.000,failed,Access denied: '../lab4_img2img/inputs/01_coas...
1,vqa_20261111_142823_490,2026-11-11T14:28:23.490430,vqa,Yes,0.464,success,None
2,caption_20261111_142823_000,2026-11-11T14:28:23.000081,caption,"The image presents a scene from a library, cha...",10.434,success,None
3,vqa_20261111_142812_518,2026-11-11T14:28:12.518927,vqa,Bicycle,0.537,success,None
4,caption_20261111_142811_936,2026-11-11T14:28:11.936835,caption,The image presents a vibrant scene set against...,12.008,success,None
5,caption_20261111_142715_144,2026-11-11T14:27:15.144778,caption,The image presents a vibrant scene set in a qu...,11.934,success,None
6,transformation_20261111_141411_158,2026-11-11T14:14:11.167846,transformation,Transformed with prompt: 'An oil painting in t...,75.337,success,None
7,vqa_20261111_141234_183,2026-11-11T14:12:34.183411,vqa,Yes,4.767,success,None
8,caption_20261111_141123_545,2026-11-11T14:11:23.545552,caption,,0.000,failed,Image file not found: '../../etc/shadow'
9,unit_test_20261111_114542_184,2026-11-11T11:45:42.184814,unit_test,Test output text,0.010,success,None


### 6. Інженерні висновки

1. **Автономність проти детермінізму:** Агентний конвеєр на базі LlamaIndex демонструє високу ефективність у багатоетапних задачах (синтез описів, модифікація підказок, виклики дифузії), проте вимагає обов'язкової наявності жорстких меж пісочниці для запобігання несанкціонованому доступу до файлової системи.
2. **Надійність валідації:** Впровадження `validate_safe_image_path` у кожен інструмент агента повністю усуває ризик атак типу Path Traversal, навіть якщо LLM формує небезпечні аргументи шляху.
3. **Персистентність:** Зв'язка реляційної бази SQLite та локального сховища артефактів гарантує повну відтворюваність історії генерацій та забезпечує швидкий аудит дій користувача і моделей.